In [2]:
pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import pandas as pd
import re
from collections import Counter

# ==========================================
# 1. CONFIGURATION
# ==========================================
# REPLACE THIS with your actual file path
FILE_PATH = r"D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\split_1_classification_all.xlsx" 
OUTPUT_FILE = r"D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\split_2_classification_vote.xlsx.xlsx"

# ==========================================
# 2. LOGIC FUNCTIONS
# ==========================================

def normalize_sdg_list(cell_value):
    if not isinstance(cell_value, str):
        return []
    
    normalized = set()
    parts = [p.strip() for p in cell_value.split(',')]
    
    for part in parts:
        part_clean = part.upper().replace(" ", "")
        
        if "NOSDG" in part_clean:
            normalized.add("NOSDG")
        else:
            match = re.search(r'(\d+)', part_clean)
            if match:
                normalized.add(f"SDG{int(match.group(1))}")
    
    return list(normalized)

def calculate_final_sdg(row):
    # Columns to check
    new_llm_cols = ['gpt_5_mini', 'gpt_5_nano', 'grok_3_mini', 'grok_4_fast', 'phi_4_mini']
    
    # --- 1. PREPARE VOTES ---
    primary_labels = normalize_sdg_list(row.get('primary_sdg', ''))
    
    new_col_votes = []
    for col in new_llm_cols:
        if col in row:
            new_col_votes.extend(normalize_sdg_list(row[col]))
    
    new_counts = Counter(new_col_votes)
    total_votes = primary_labels + new_col_votes
    total_counts = Counter(total_votes)
    
    final_set = set()
    
    # --- 2. DETERMINE TRUST LEVEL ---
    # High Trust = Primary is Human Labelled (SDG 1-16)
    is_high_trust = False
    if primary_labels and primary_labels[0] not in ['NOSDG', 'SDG17']:
        try:
            num = int(primary_labels[0].replace('SDG',''))
            if 1 <= num <= 16:
                is_high_trust = True
        except:
            pass

    # --- 3. APPLY RULES ---
    
    if is_high_trust:
        # === HIGH TRUST LOGIC (Human 1-16) ===
        final_set.update(primary_labels)
        
        # Add others if >= 2 votes in NEW columns
        for sdg, count in new_counts.items():
            if count >= 2 and sdg != "NOSDG":
                final_set.add(sdg)
                
        # Mutual Exclusivity: If we have a Real SDG, kill NOSDG
        if "NOSDG" in final_set:
            final_set.remove("NOSDG")

    else:
        # === LOW TRUST LOGIC (Primary is NOSDG or SDG17) ===
        # Threshold: Need >= 3 votes TOTAL to play
        candidates = [sdg for sdg, count in total_counts.items() if count >= 3]
        
        real_sdgs = [x for x in candidates if x != "NOSDG"]
        nosdg_votes = total_counts.get("NOSDG", 0)
        
        # Calculate strongest Real SDG votes
        max_real_votes = 0
        if real_sdgs:
            max_real_votes = max([total_counts[x] for x in real_sdgs])
        
        # DOMINANCE CHECK
        if "NOSDG" in candidates:
            # UPDATED RULE: Real SDG must be STRICTLY GREATER than NOSDG to win.
            # If Tie (3 vs 3) -> NOSDG wins.
            if max_real_votes > nosdg_votes and real_sdgs:
                 final_set.update(real_sdgs)
            else:
                 return "NOSDG" # NOSDG wins ties or outright majority
        else:
            # NOSDG didn't reach threshold (votes < 3)
            if real_sdgs:
                final_set.update(real_sdgs)
            else:
                return "NOSDG"

    # --- 4. FORMATTING ---
    if not final_set:
        return "NOSDG"
        
    def sort_key(s):
        return int(s.replace("SDG", ""))
        
    return ", ".join(sorted(list(final_set), key=sort_key))

# ==========================================
# 3. EXECUTION
# ==========================================
try:
    print(f"Reading file: {FILE_PATH}...")
    df = pd.read_excel(FILE_PATH)
    
    df.columns = df.columns.str.strip() # Clean headers

    print("Calculating Final SDGs...")
    df['final_sdg'] = df.apply(calculate_final_sdg, axis=1)

    print(f"Saving to {OUTPUT_FILE}...")
    df.to_excel(OUTPUT_FILE, index=False)
    print("Success.")

except Exception as e:
    print(f"Error: {e}")

Reading file: D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\split_1_classification_all.xlsx...
Calculating Final SDGs...
Saving to D:\Stevens\003 PhD\Fall 25\AAI 646\Final_Project\code\split_1_classification_vode2.xlsx.xlsx...
Success.
